# Flood Mapping and Vulnerability Assessment in River and Lagoon Environments

*by Daniele Petrini & Federico Urbini*

## Necessary Imports

In [ ]:
# Import libraries and project modules.

import ee
import geemap
import pandas as pd
from pathlib import Path
import sys
import importlib
from IPython.display import display

from utils import config as cfg
from utils import utils_SAR as su
from utils import utils_setup as us
from utils import utils_optical as ou

importlib.reload(cfg)
importlib.reload(us)
importlib.reload(su)
importlib.reload(ou)

/home/federico/miniconda3/envs/floods/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


<module 'utils.utils_optical' from '/home/federico/Desktop/floods-SAR/utils/utils_optical.py'>

## Authentication

In [2]:
ee.Authenticate(auth_mode="localhost")
ee.Initialize(project="floods-2024")
print("Earth Engine initialized correctly.")


Earth Engine initialized correctly.


## Output Folders

In [3]:
# Create output folders

Path("outputs").mkdir(exist_ok=True)
Path("outputs/tables").mkdir(parents=True, exist_ok=True)
Path("outputs/figures").mkdir(parents=True, exist_ok=True)
Path("outputs/maps").mkdir(parents=True, exist_ok=True)

## Flags

In [14]:
# Define SAR notebook flags.

MONTH = "sept" # Options: "sept", "oct"
MAKE_MAPS = True

SAR_OUTPUT_DIR = Path("outputs") / "tables"
SAR_MAP_DIR = Path("outputs") / "maps"

MAKE_OPTICAL_MAP = True

OPTICAL_OUTPUT_DIR = Path("outputs") / "tables"
OPTICAL_MAP_DIR = Path("outputs") / "maps"

## Load Datasets

In [5]:
# Load project configuration and auxiliary datasets.

project = cfg.get_project_config()

aoi_emilia_romagna = project["aoi_emilia_romagna"]
places = project["places"]
dates = project["dates"]
events = project["events"]

auxiliary = us.load_auxiliary_datasets(aoi_emilia_romagna)

permanent_water = auxiliary["permanent_water"]
slope = auxiliary["slope"]
rivers_image = auxiliary["rivers_image"]

vis = su.get_sar_vis_params()

print("Configuration and auxiliary datasets loaded.")

Configuration and auxiliary datasets loaded.


## SAR

In [6]:
# Run SAR flood mapping for the selected event.

results = su.run_sar_event(
    event_key=MONTH,
    event=events[MONTH],
    dates=dates,
    permanent_water=permanent_water,
    slope=slope,
    rivers_image=rivers_image,
    places=places,
    vis=vis,
    output_dir=SAR_OUTPUT_DIR,
    make_map=MAKE_MAPS,
    **cfg.SAR_PARAMS[MONTH],
)

display(results["df_candidates"])
display(results["df_metadata"])
display(results["df_area"])

print("Selected Sentinel-1 geometry:")
print(results["selected"])

print("Otsu threshold:", results["otsu_threshold"])
print("Otsu physically valid:", results["otsu_valid"])

,orbit_pass,relative_orbit,pre_count,post_count,balanced_count,total_count
0,ASCENDING,15,2,1,1,3
1,ASCENDING,44,2,1,1,3
2,DESCENDING,168,2,1,1,3


,date,label,mean_incidence_angle,orbit_pass,polarizations,relative_orbit,resolution_meters,system_index
0,2024-09-18 05:27,sept_post,31.164600,DESCENDING,VV/VH,168,10,S1A_IW_GRDH_1SDV_20240918T052757_20240918T0528...
1,2024-09-19 17:14,sept_post,45.864226,ASCENDING,VV/VH,15,10,S1A_IW_GRDH_1SDV_20240919T171453_20240919T1715...
2,2024-09-21 16:58,sept_post,30.670397,ASCENDING,VV/VH,44,10,S1A_IW_GRDH_1SDV_20240921T165840_20240921T1659...
3,2024-08-20 05:19,sept_pre,39.518365,DESCENDING,VV/VH,95,10,S1A_IW_GRDH_1SDV_20240820T051947_20240820T0520...
4,2024-08-21 17:06,sept_pre,38.782118,ASCENDING,VV/VH,117,10,S1A_IW_GRDH_1SDV_20240821T170631_20240821T1706...
5,2024-08-21 17:06,sept_pre,38.543054,ASCENDING,VV/VH,117,10,S1A_IW_GRDH_1SDV_20240821T170656_20240821T1707...
6,2024-08-25 05:27,sept_pre,31.204167,DESCENDING,VV/VH,168,10,S1A_IW_GRDH_1SDV_20240825T052756_20240825T0528...
7,2024-08-26 17:14,sept_pre,45.864466,ASCENDING,VV/VH,15,10,S1A_IW_GRDH_1SDV_20240826T171452_20240826T1715...
8,2024-08-28 16:58,sept_pre,30.688373,ASCENDING,VV/VH,44,10,S1A_IW_GRDH_1SDV_20240828T165839_20240828T1659...
9,2024-09-02 17:06,sept_pre,38.781936,ASCENDING,VV/VH,117,10,S1A_IW_GRDH_1SDV_20240902T170631_20240902T1706...


,event,method,used_in_confidence,area_km2
0,sept,delta_VV_lt_-2.5_dB,True,38.295058
1,sept,ratio_lt_0.7,True,72.125182
2,sept,VV_post_lt_-17_dB,True,26.473151
3,sept,otsu_delta_VV_lt_-1.484_dB,False,75.023261
4,sept,sar_confidence_medium_high,True,44.471016
5,sept,sar_confidence_high,True,11.971913


Selected Sentinel-1 geometry:
{'selection_mode': 'manual_override', 'orbit_pass': 'ASCENDING', 'relative_orbit': 44, 'pre_image_count': 2, 'post_image_count': 1}
Otsu threshold: -1.4838502626889183
Otsu physically valid: True


In [23]:
# Display and save SAR map as an interactive HTML file.

display(results["map"])
map_path = SAR_MAP_DIR / f"{MONTH}_sar_map.html"
results["map"].to_html(str(map_path))
print(f"Map saved to: {map_path}")


Map(bottom=3041131.0, center=[44.29046580838405, 12.134571075439455], controls=(WidgetControl(options=['positi…

Map saved to: outputs/maps/sept_sar_map.html


In [8]:
# Save SAR area summary.

df_sar_area_summary = results["df_area"].copy()

display(df_sar_area_summary)

df_sar_area_summary.to_csv(
    SAR_OUTPUT_DIR / f"{MONTH}_sar_area_summary.csv",
    index=False,
)

,event,method,used_in_confidence,area_km2
0,sept,delta_VV_lt_-2.5_dB,True,38.295058
1,sept,ratio_lt_0.7,True,72.125182
2,sept,VV_post_lt_-17_dB,True,26.473151
3,sept,otsu_delta_VV_lt_-1.484_dB,False,75.023261
4,sept,sar_confidence_medium_high,True,44.471016
5,sept,sar_confidence_high,True,11.971913


In [9]:
# Save selected Sentinel-1 geometry summary.

df_selected_s1 = pd.DataFrame([
    {
        "event": MONTH,
        **results["selected"],
        "otsu_threshold_delta_vv": results["otsu_threshold"],
        "otsu_valid": results["otsu_valid"],
    }
])

display(df_selected_s1)

df_selected_s1.to_csv(
    SAR_OUTPUT_DIR / f"{MONTH}_selected_s1_geometry_summary.csv",
    index=False,
)

,event,selection_mode,orbit_pass,relative_orbit,pre_image_count,post_image_count,otsu_threshold_delta_vv,otsu_valid
0,sept,manual_override,ASCENDING,44,2,1,-1.48385,True


# OPTICAL

In [10]:
# Run Sentinel-2 optical processing and SAR-optical fusion for the selected event.

opt_vis = ou.get_optical_vis_params()

optical_results = ou.run_optical_event(
    event_key=MONTH,
    event=events[MONTH],
    dates=dates,
    sar_confidence=results["sar_confidence"],
    permanent_water=permanent_water,
    places=places,
    output_dir=OPTICAL_OUTPUT_DIR,
    make_map=MAKE_OPTICAL_MAP,
    vis=opt_vis,
    **cfg.OPTICAL_PARAMS[MONTH],
)

display(optical_results["df_s2_metadata"])
display(optical_results["df_valid_summary"])
display(optical_results["df_agreement"])

,cloudy_pixel_percentage,date,label,mgrs_tile,system_index
0,92.319429,2024-09-19 10:18,sept_post,32TPP,20240919T101019_20240919T101702_T32TPP
1,99.147075,2024-09-19 10:18,sept_post,32TPQ,20240919T101019_20240919T101702_T32TPQ
2,99.967992,2024-09-19 10:18,sept_post,32TQP,20240919T101019_20240919T101702_T32TQP
3,99.999452,2024-09-19 10:18,sept_post,32TQQ,20240919T101019_20240919T101702_T32TQQ
4,99.997205,2024-09-19 10:18,sept_post,33TUJ,20240919T101019_20240919T101702_T33TUJ
...,...,...,...,...,...
63,93.543178,2024-09-14 10:18,sept_pre,32TPQ,20240914T100631_20240914T100718_T32TPQ
64,42.533979,2024-09-14 10:18,sept_pre,32TQP,20240914T100631_20240914T100718_T32TQP
65,89.395469,2024-09-14 10:18,sept_pre,32TQQ,20240914T100631_20240914T100718_T32TQQ
66,61.121106,2024-09-14 10:18,sept_pre,33TUJ,20240914T100631_20240914T100718_T33TUJ


,event,window,valid_pixel_percentage_aoi
0,sept,pre,99.882410
1,sept,post,34.142835


,sar_area_km2_valid_optical,optical_area_km2,intersection_km2,union_km2,iou,dice,event,optical_index,optical_threshold
0,32.100245,50.834384,9.800303,69.749637,0.140507,0.236338,sept,MNDWI,0.0


In [24]:
# Display optical and SAR-optical fusion map.

display(optical_results["map"])
map_path = OPTICAL_MAP_DIR / f"{MONTH}_optical_map.html"
optical_results["map"].to_html(str(map_path))
print(f"Map saved to: {map_path}")

Map(bottom=95289.0, center=[44.326795363769975, 11.87347412109375], controls=(WidgetControl(options=['position…

Map saved to: outputs/maps/sept_optical_map.html


In [12]:
# Save SAR-optical agreement summary.

df_sar_optical_summary = optical_results["df_agreement"].copy()

display(df_sar_optical_summary)

df_sar_optical_summary.to_csv(
    OPTICAL_OUTPUT_DIR / f"{MONTH}_sar_optical_agreement_summary.csv",
    index=False,
)

,sar_area_km2_valid_optical,optical_area_km2,intersection_km2,union_km2,iou,dice,event,optical_index,optical_threshold
0,32.100245,50.834384,9.800303,69.749637,0.140507,0.236338,sept,MNDWI,0.0
